# 5.2 端侧部署框架

端侧部署框架将优化后的模型部署到具体硬件上，解决跨平台兼容性、算子支持度和运行时调度等问题。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import struct
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### GGUF格式模拟 (llama.cpp生态)

GGUF是llama.cpp的标准模型格式，支持多种量化类型，使用mmap零拷贝加载。

In [ ]:
class GGUFQuantizationSimulator:
    """GGUF量化格式模拟器"""
    QUANT_TYPES = {
        'Q4_0': {'bits': 4, 'group_size': 32, 'desc': '4-bit量化, block大小32'},
        'Q4_1': {'bits': 4, 'group_size': 32, 'desc': '4-bit量化+scale+dmin'},
        'Q5_0': {'bits': 5, 'group_size': 32, 'desc': '5-bit量化'},
        'Q5_1': {'bits': 5, 'group_size': 32, 'desc': '5-bit量化+scale+dmin'},
        'Q8_0': {'bits': 8, 'group_size': 32, 'desc': '8-bit量化, block大小32'},
        'Q4_K_M': {'bits': 4, 'group_size': 128, 'desc': '4-bit k-quant混合, 中等质量'},
        'Q5_K_M': {'bits': 5, 'group_size': 128, 'desc': '5-bit k-quant混合, 中等质量'},
        'Q6_K': {'bits': 6, 'group_size': 256, 'desc': '6-bit k-quant, 高质量'},
    }

    @staticmethod
    def estimate_model_size(n_params_b, quant_type):
        """估算量化后模型大小"""
        info = GGUFQuantizationSimulator.QUANT_TYPES[quant_type]
        bits = info['bits']
        group_size = info['group_size']
        weight_bytes = n_params_b * 1e9 * bits / 8
        scale_bytes = n_params_b * 1e9 / group_size * 2
        total_gb = (weight_bytes + scale_bytes) / 1e9
        return total_gb

    @staticmethod
    def simulate_quantize(weight, quant_type='Q4_0'):
        """模拟GGUF量化"""
        info = GGUFQuantizationSimulator.QUANT_TYPES[quant_type]
        bits = info['bits']
        group_size = info['group_size']
        q_max = 2 ** (bits - 1) - 1
        q_min = -2 ** (bits - 1)

        orig_shape = weight.shape
        n_groups = weight.numel() // group_size
        w_grouped = weight.reshape(n_groups, group_size)
        scale = w_grouped.abs().amax(dim=-1, keepdim=True) / q_max
        scale = torch.clamp(scale, min=1e-8)
        w_q = torch.clamp(torch.round(w_grouped / scale), q_min, q_max)
        w_deq = (w_q * scale).reshape(orig_shape)
        return w_deq

weight = torch.randn(1024, 1024)
print(f"=== GGUF量化格式对比 ===")
print(f"\n{'格式':<10} {'位数':<6} {'MSE':<12} {'7B模型大小(GB)':<18}")
print("-" * 46)

for qtype, info in GGUFQuantizationSimulator.QUANT_TYPES.items():
    w_deq = GGUFQuantizationSimulator.simulate_quantize(weight, qtype)
    mse = torch.nn.functional.mse_loss(weight, w_deq)
    size_gb = GGUFQuantizationSimulator.estimate_model_size(7, qtype)
    print(f"{qtype:<10} {info['bits']}bit   {mse:<12.6f} {size_gb:<18.2f}")

print(f"\nllama.cpp核心优势:")
print(f"1. 纯C/C++实现, 零依赖")
print(f"2. mmap加载, 零拷贝")
print(f"3. 丰富的量化格式")
print(f"4. 跨平台(CPU优先)")

### ExecuTorch模型导出流程模拟

In [ ]:
class ExecuTorchSimulator:
    """ExecuTorch导出流程模拟"""
    def __init__(self):
        self.export_steps = [
            ('1. Export', 'torch.export导出模型为EdgeDialect'),
            ('2. To Edge', '转换为EdgeDialect IR'),
            ('3. Partition', '按后端分区(NPU/CPU/DSP)'),
            ('4. Compile', '为每个分区生成优化代码'),
            ('5. Bundle', '打包为pte文件'),
        ]

    def simulate_export(self, model, example_input):
        """模拟ExecuTorch导出流程
        注意：本实现为教学演示，仅展示导出步骤和估算文件大小，
        未执行真实的torch.export或pte文件生成。"""
        print(f"=== ExecuTorch导出流程 ===")
        for step_name, desc in self.export_steps:
            print(f"  {step_name}: {desc}")

        with torch.no_grad():
            out = model(example_input)

        params = sum(p.numel() for p in model.parameters())
        pte_size = params * 0.5 / 1024 / 1024

        return {
            'output_shape': out.shape,
            'n_params': params,
            'estimated_pte_size_mb': pte_size,
        }

class SimpleLLMForExport(nn.Module):
    def __init__(self, dim=128, n_layers=4, n_heads=4):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True)
            for _ in range(n_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = SimpleLLMForExport(dim=128, n_layers=4, n_heads=4)
x = torch.randn(1, 16, 128)

executorch = ExecuTorchSimulator()
result = executorch.simulate_export(model, x)

print(f"\n导出结果:")
print(f"  输出形状: {result['output_shape']}")
print(f"  参数量: {result['n_params']:,}")
print(f"  估计pte大小: {result['estimated_pte_size_mb']:.1f} MB (INT4)")

print(f"\n=== 端侧部署框架选择指南 ===")
frameworks = [
    ('llama.cpp/GGUF', 'CPU推理', '跨平台,零依赖,社区活跃'),
    ('MLC-LLM', 'CPU/GPU/NPU', 'TVM编译,原生代码'),
    ('ExecuTorch', 'iOS/Android', 'PyTorch生态,官方支持'),
    ('ONNX Runtime', 'CPU/GPU/NPU', '通用推理,成熟稳定'),
    ('Core ML', 'Apple平台', 'ANE加速,生态集成'),
    ('NCNN/MNN', 'ARM CPU', '极致ARM优化'),
]
print(f"\n{'框架':<18} {'目标硬件':<15} {'特点':<30}")
print("-" * 63)
for name, hw, feature in frameworks:
    print(f"{name:<18} {hw:<15} {feature:<30}")

### ONNX导出与推理

ONNX（Open Neural Network Exchange）是跨框架模型交换的标准格式，可将PyTorch模型导出后在ONNX Runtime上高效推理。

In [ ]:
import io
import os

class SimpleModelForONNX(nn.Module):
    """用于ONNX导出测试的简单模型"""
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)

onnx_model = SimpleModelForONNX(dim=64, n_classes=10)
onnx_model.eval()
dummy_input = torch.randn(1, 64)

onnx_buffer = io.BytesIO()
torch.onnx.export(
    onnx_model,
    dummy_input,
    onnx_buffer,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=14,
)
onnx_size = onnx_buffer.tell()
onnx_buffer.seek(0)

with torch.no_grad():
    pt_output = onnx_model(dummy_input)

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_buffer.getvalue())
    ort_output = sess.run(None, {'input': dummy_input.numpy()})[0]
    max_diff = np.abs(pt_output.numpy() - ort_output).max()
    onnx_runtime_available = True
except ImportError:
    max_diff = 0.0
    onnx_runtime_available = False

print(f"=== ONNX导出与推理 ===")
print(f"模型参数量: {sum(p.numel() for p in onnx_model.parameters()):,}")
print(f"ONNX文件大小: {onnx_size / 1024:.1f} KB")
print(f"PyTorch输出范围: [{pt_output.min():.4f}, {pt_output.max():.4f}]")
if onnx_runtime_available:
    print(f"ONNX Runtime输出范围: [{ort_output.min():.4f}, {ort_output.max():.4f}]")
    print(f"最大数值差异: {max_diff:.8f}")
    print(f"ONNX Runtime推理: 成功")
else:
    print(f"ONNX Runtime未安装，跳过推理验证")
    print(f"安装方法: pip install onnxruntime")

print(f"\nONNX导出关键参数:")
print(f"  opset_version: 14 (推荐14+)")
print(f"  dynamic_axes: 支持动态batch")
print(f"  常见问题: 自定义算子不支持 → 需注册自定义ONNX算子")

### 模型转换与部署流程对比

不同部署框架的模型转换流程和适用场景对比。

In [ ]:
print(f"=== 模型转换与部署流程对比 ===")
print()
workflows = [
    {
        'name': 'llama.cpp / GGUF',
        'steps': ['PyTorch → convert-hf-to-gguf.py → GGUF文件', 'llama-cli / llama-server 加载推理'],
        'pros': '零依赖, CPU优先, mmap加载, 社区量化格式丰富',
        'cons': '仅CPU推理(部分GPU), 无NPU加速',
        'best_for': '通用CPU部署, 快速原型验证',
    },
    {
        'name': 'ExecuTorch',
        'steps': ['torch.export → to_edge → partition → compile → .pte文件', 'ExecuTorch runtime加载推理'],
        'pros': 'PyTorch生态无缝, 支持NPU/DSP后端',
        'cons': '导出限制多(动态shape/自定义算子), 生态仍在建设',
        'best_for': 'PyTorch用户, Android/iOS端侧',
    },
    {
        'name': 'ONNX Runtime',
        'steps': ['torch.onnx.export → .onnx文件', 'onnxruntime.InferenceSession加载推理'],
        'pros': '跨平台, 算子标准, NPU/GPU delegate',
        'cons': '动态shape支持有限, LLM支持不如llama.cpp',
        'best_for': '通用推理, 非LLM模型部署',
    },
    {
        'name': 'Core ML',
        'steps': ['coremltools.convert → .mlpackage', 'Core ML framework加载推理'],
        'pros': 'ANE加速, iOS/macOS深度集成, 隐私友好',
        'cons': '仅Apple平台, 算子覆盖有限',
        'best_for': 'iOS/macOS应用, Apple Silicon优化',
    },
]

for wf in workflows:
    print(f"--- {wf['name']} ---")
    print(f"  转换流程: {' → '.join(wf['steps'])}")
    print(f"  优势: {wf['pros']}")
    print(f"  劣势: {wf['cons']}")
    print(f"  最佳场景: {wf['best_for']}")
    print()

print(f"=== 端侧部署常见问题 ===")
issues = [
    ('动态shape', 'LLM序列长度动态变化 → 固定max_seq_len或使用dynamic axes'),
    ('自定义算子', 'RoPE/SwiGLU等算子可能不支持 → 手动注册或拆分为基础算子'),
    ('量化格式', '不同框架量化格式不兼容 → 使用框架自带量化工具'),
    ('NPU兼容性', 'NPU算子覆盖有限 → 不支持的算子回退CPU'),
    ('内存限制', '端侧内存有限 → 权重按需加载+KV Cache优化'),
]
for issue, solution in issues:
    print(f"  {issue}: {solution}")